# NSE オーナー企業分析

`nse_full_download.ipynb` が構築した `data/cache/nse/nse_index.db` を分析し、
`data/exports/nse/` 配下に以下の 4 ファイルを出力する。

| 出力                         | 内容                                                                        |
| ---------------------------- | --------------------------------------------------------------------------- |
| `shareholdings.csv`          | shareholdings テーブル（symbol/company_name/isin + 集約 promoter/public %） |
| `shareholding_detail.csv`    | shareholding_detail テーブル（XBRL 詳細株主データ）                         |
| `owner_candidates.csv`       | 緩フィルタ適用 + `owner_flag` ラベル付与                                    |
| `xbrl_category_reference.md` | XBRL カテゴリ/サブカテゴリ分類体系レポート                                  |

## 実行フロー

1. Section 1〜3 を実行 → 上記 4 ファイル生成
2. Claude Code にアドホック依頼して `owner_candidates.csv` の AI 列を埋める
3. Section 4 を実行 → `owner_flag_final` 確定
4. Section 5 で結果確認

## 前提

- `nse_full_download.ipynb` の Phase 1〜4 完了済み
- DB: `data/cache/nse/nse_index.db`

## owner_flag ラベル体系

| Tier | ラベル                                    | 条件 (要約)                                          | AI review |
| ---- | ----------------------------------------- | ---------------------------------------------------- | --------- |
| 1    | `owner_confirmed_individual_and_director` | hufi≥1 AND (dir≥1 OR kmp≥1) AND foreign<50           | 不要      |
| 1    | `owner_confirmed_individual`              | hufi≥1 AND foreign<50                                | 不要      |
| 1    | `owner_confirmed_director_only`           | (dir≥1 OR kmp≥1) AND hufi=0 AND nri=0 AND foreign<50 | 不要      |
| 2    | `owner_probable_nri_family`               | nri≥1 AND hufi=0                                     | 対象      |
| 2    | `owner_probable_relatives_trust`          | (rel≥1 OR trust≥1) AND 他自然人=0                    | 対象      |
| 3    | `ambiguous_mnc_jv_candidate`              | hufi≥1 AND foreign≥50                                | 対象      |
| 3    | `ambiguous_minor_individual`              | hufi≥1 AND hufi_pct<0.5 AND dir=kmp=0                | 対象      |
| 3    | `ambiguous_holding_indian`                | natural=0 AND other_indian≥10                        | 対象      |
| 3    | `ambiguous_holding_foreign`               | natural=0 AND other_foreign≥10                       | 対象      |
| 4    | `excluded_low_promoter`                   | promoter<10                                          | 不要      |
| 4    | `excluded_state_dominant`                 | govt≥10                                              | 不要      |
| 4    | `excluded_no_natural_no_holding`          | 上記いずれにも該当せず                               | 不要      |


In [ ]:
# Cell 2: Imports
from __future__ import annotations

import sqlite3
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

print("Imports OK")


In [ ]:
# Cell 3: Config

# 入出力パス
DB_PATH: Path = Path("data/cache/nse/nse_index.db")
EXPORT_DIR: Path = Path("data/exports/nse")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

OUT_SHAREHOLDINGS: Path = EXPORT_DIR / "shareholdings.csv"
OUT_SHAREHOLDING_DETAIL: Path = EXPORT_DIR / "shareholding_detail.csv"
OUT_OWNER_CANDIDATES: Path = EXPORT_DIR / "owner_candidates.csv"
OUT_XBRL_REFERENCE: Path = EXPORT_DIR / "xbrl_category_reference.md"

# Owner フィルタ閾値
MIN_PROMOTER_PCT: float = 10.0
MAX_FOREIGN_NON_GOVT_PCT: float = 50.0    # MNC-JV 除外の境界
MAX_GOVT_DOMINANT_PCT: float = 10.0       # PSU 除外の境界
MIN_HOLDING_PCT: float = 10.0             # holding 経由型の閾値
MIN_MINOR_HUFI_PCT: float = 0.5           # 微小個人と通常個人の境界

# promoter_names_full_list の区切り文字
NAMES_SEPARATOR: str = "|"

# AI review 対象ラベル
AI_REVIEW_LABELS: set[str] = {
    "owner_probable_nri_family",
    "owner_probable_relatives_trust",
    "ambiguous_mnc_jv_candidate",
    "ambiguous_minor_individual",
    "ambiguous_holding_indian",
    "ambiguous_holding_foreign",
    "ambiguous_director_only_unknown",  # act-2026-04-30-004 hybrid rule
}

print(f"DB_PATH             = {DB_PATH}")
print(f"EXPORT_DIR          = {EXPORT_DIR}")
print(f"MIN_PROMOTER_PCT    = {MIN_PROMOTER_PCT}%")
print(f"MAX_FOREIGN_NON_GOVT= {MAX_FOREIGN_NON_GOVT_PCT}%")
print(f"MAX_GOVT_DOMINANT   = {MAX_GOVT_DOMINANT_PCT}%")


## Section 1: 既存 CSV 退避 + テーブル再出力

既存の `shareholdings.csv` / `shareholding_detail.csv` を `_` prefix でリネーム退避し、
stocks テーブルと JOIN した新形式 (symbol, company_name, isin + 既存列) で再出力する。


In [ ]:
# Cell 5: 既存 CSV を _ prefix で退避
for name in ["shareholdings.csv", "shareholding_detail.csv", "owner_candidates.csv"]:
    src = EXPORT_DIR / name
    dst = EXPORT_DIR / f"_{name}"
    if src.exists():
        if dst.exists():
            dst.unlink()
        src.rename(dst)
        print(f"  退避: {name} → _{name}")
    else:
        print(f"  (skip) {name} は存在しません")


In [ ]:
# Cell 6: shareholdings.csv を stocks JOIN で出力
conn = sqlite3.connect(DB_PATH)

shareholdings = pd.read_sql_query("""
    SELECT
        sh.symbol,
        s.company_name,
        s.isin,
        sh.as_on_date,
        sh.promoter_pct,
        sh.public_pct,
        sh.employee_trust_pct,
        sh.submission_date,
        sh.broadcast_date,
        sh.xbrl_url,
        sh.fetched_at
    FROM shareholdings sh
    LEFT JOIN stocks s ON s.symbol = sh.symbol
    ORDER BY sh.symbol, sh.as_on_date DESC
""", conn)

shareholdings.to_csv(OUT_SHAREHOLDINGS, index=False, encoding="utf-8-sig")
print(f"出力: {OUT_SHAREHOLDINGS}  ({len(shareholdings):,} 行)")
print(f"  unique symbols: {shareholdings['symbol'].nunique():,}")
display(shareholdings.head(3))  # noqa: F821


In [ ]:
# Cell 7: shareholding_detail.csv を stocks JOIN で出力
shareholding_detail = pd.read_sql_query("""
    SELECT
        d.symbol,
        s.company_name,
        s.isin,
        d.report_date,
        d.category,
        d.sub_category,
        d.shareholder_name,
        d.pan,
        d.num_shareholders,
        d.num_fully_paid_shares,
        d.num_voting_rights,
        d.pct_total_shares,
        d.pct_fully_diluted,
        d.num_shares_demat,
        d.is_category_total,
        d.fetched_at
    FROM shareholding_detail d
    LEFT JOIN stocks s ON s.symbol = d.symbol
    ORDER BY d.symbol, d.report_date DESC, d.category, d.sub_category
""", conn)

shareholding_detail.to_csv(OUT_SHAREHOLDING_DETAIL, index=False, encoding="utf-8-sig")
print(f"出力: {OUT_SHAREHOLDING_DETAIL}  ({len(shareholding_detail):,} 行)")
print(f"  unique symbols: {shareholding_detail['symbol'].nunique():,}")
display(shareholding_detail.head(3))  # noqa: F821


## Section 2: XBRL カテゴリ分類レポート生成

`src/market/nse/xbrl.py` で使用している XBRL taxonomy のカテゴリ・サブカテゴリ体系を
Markdown で出力する。`owner_flag` 判定ロジックとの対応表も含む。


In [ ]:
# Cell 9: xbrl_category_reference.md を生成
from datetime import date

_today = date.today().isoformat()
_content = "# NSE/BSE XBRL Shareholding Pattern カテゴリ/サブカテゴリ分類体系\n\n**生成日**: {today}\n**対象 taxonomy**: BSE SHP XBRL (2018-03-31 / 2022-09-30 / 2025-05-31 / 2025-10-31)\n**出典**:\n- SEBI (ICDR) Regulations 2009/2018 (sebi.gov.in)\n- SEBI (LODR) Regulations 2015 (sebi.gov.in)\n- Companies Act 2013 (indiacode.nic.in)\n- BSE XBRL Taxonomy (bseindia.com/xbrl/)\n\n---\n\n## 目次\n\n1. [データ構造](#1-データ構造)\n2. [Top-level Category](#2-top-level-category)\n3. [PromoterAndPromoterGroup サブカテゴリ](#3-promoterandpromotergroup-サブカテゴリ)\n4. [PublicShareholding サブカテゴリ](#4-publicshareholding-サブカテゴリ)\n5. [NonPromoterNonPublic サブカテゴリ](#5-nonpromoternonpublic-サブカテゴリ)\n6. [is_category_total フラグの意味](#6-is_category_total-フラグの意味)\n7. [Filers の慣習・注意点](#7-filers-の慣習注意点)\n8. [Owner 判定ロジック対応表](#8-owner-判定ロジック対応表)\n9. [出典（1 次情報）](#9-出典1-次情報)\n\n---\n\n## 1. データ構造\n\n`shareholding_detail.csv` は 3 階層構造:\n\n- **`category`** (top-level): 3 種 (Promoter / Public / NonPromoterNonPublic)\n- **`sub_category`**: 約 60 種の細分類\n- **`shareholder_name`**: 個別株主名 (is_category_total=0 の明細行のみ)\n\n1 銘柄 × 1 四半期あたり、集計行と個別明細行が合わせて 50〜200 行出力される。\n\n---\n\n## 2. Top-level Category\n\n| category | 日本語 | 意味 |\n|---|---|---|\n| `PromoterAndPromoterGroup` | Promoter and Promoter Group | 創業者・支配株主・その親族・関連会社 |\n| `PublicShareholding` | Public Shareholding | 一般公開株主（機関投資家・個人投資家） |\n| `NonPromoterNonPublic` | Non-Promoter Non-Public | 上記以外（信託・Custodian 等） |\n\n---\n\n## 3. PromoterAndPromoterGroup サブカテゴリ\n\nSEBI (ICDR) 2009 Reg 2(1)(zb) の promoter group 定義に対応する XBRL 分類。\n\n### 3.1 自然人系 (Natural Persons)\n\n本人・親族・取締役・KMP・ファミリートラスト。**Owner 判定の核**。\n\n| sub_category (XBRL) | 日本語 | SEBI Table II 行 |\n|---|---|---|\n| `IndividualsOrHinduUndividedFamily` | 個人・Hindu Undivided Family | **A(1)(a)** |\n| `NonResidentIndividualsOrForeignIndividuals` | NRI・外国人個人 | **A(2)(a)** |\n| `DirectorsAndDirectorsRelatives` | 取締役と親族 | A(1)(d) 内訳 |\n| `KeyManagerialPersonnel` | Key Managerial Personnel (CEO/CFO/CS) | A(1)(d) 内訳 |\n| `RelativesOfPromotersOtherThanPromoterGroup` | 狭義 promoter group 外の親族 | A(1)(d) 内訳 |\n| `TrustsWhereAnyPersonBelongingToPromoterAndPromoterGroupIsisTrusteeOrBeneficiaryOrAuthorOfTrust` | ファミリートラスト | A(1)(d) 内訳 |\n\n### 3.2 法人系 (Bodies Corporate)\n\n| sub_category (XBRL) | 日本語 |\n|---|---|\n| `AssociateCompaniesOrSubsidiaries` | 関連会社・子会社 |\n| `BodiesCorporateIncludingSubsidiaries` | 法人（子会社含む） |\n| `OtherIndianShareholders` | その他 Indian 株主（家族 holding company が多い） |\n| `OtherForeignShareholders` | その他 Foreign 株主（外国 holding company 等） |\n| `IndianFinancialInstitutionsOrBanks` | Indian 金融機関・銀行 |\n\n### 3.3 政府系 (Government)\n\n| sub_category (XBRL) | 日本語 |\n|---|---|\n| `CentralGovernmentOrPresidentOfIndia` | 中央政府・大統領 |\n| `StateGovernmentsOrGovernors` | 州政府・知事 |\n| `ForeignGovernment` | 外国政府 |\n| `ShareholdingByCompaniesOrBodiesCorporatewhereCentralOrStateGovernmentIsPromoter` | 政府系 body corporate |\n| `CentralGovernmentOrStateGovernmentS` | 中央・州政府（旧 taxonomy） |\n\n### 3.4 外国系 Institutions / FPI (promoter 側)\n\n| sub_category (XBRL) | 日本語 |\n|---|---|\n| `ForeignInstitutions` | 外国機関 |\n| `ForeignPortfolioInvestor` | FPI |\n\n### 3.5 再集計行 (⚠️ 二重計上注意)\n\nこれらは他 sub_category の**小計**。素朴に SUM すると二重計上になる。\n\n| sub_category | 集計対象 |\n|---|---|\n| `Indian` | Individuals/HUF + Governments + Banks + OtherIndian 等の合算 |\n| `Foreign` | NRI + Foreign Govt + Foreign Inst + FPI + Other Foreign 等の合算 |\n| `Governments` / `Goverments` | 中央 + 州 + Foreign 政府の合算（綴り違いは taxonomy 版依存）|\n| `CentralAndStateGovernments` | 中央 + 州政府の合算 |\n| `ForeignPromotersCumulativeOrAggregation` | 外国 promoter の cumulative |\n\n**集計時のルール**: 個別 sub_category を明示列挙して SUM すること。\n\n---\n\n## 4. PublicShareholding サブカテゴリ\n\n### 4.1 国内機関投資家 (Institutions Domestic)\n\n| sub_category | 日本語 |\n|---|---|\n| `MutualFundsOrUti` | Mutual Funds / UTI |\n| `VentureCapitalFunds` | VC |\n| `AlternativeInvestmentFunds` | AIF |\n| `Banks` | 銀行 |\n| `InsuranceCompanies` | 保険会社 |\n| `ProvidentFundsOrPensionFunds` | 年金基金 |\n| `AssetReconstructionCompanies` | ARC |\n| `SovereignWealthFundsDomestic` | 国内 SWF |\n| `NBFCsRegisteredWithRbi` | NBFC (RBI 登録) |\n| `OtherFinancialInstitutions` | その他金融機関 |\n| `OtherInstitutionsDomestic` | その他国内機関 |\n| `InstitutionsDomestic` | **（再集計）** 国内機関合計 |\n\n### 4.2 外国機関投資家 (Institutions Foreign)\n\n| sub_category | 日本語 |\n|---|---|\n| `ForeignDirectInvestment` | FDI |\n| `ForeignVentureCapitalInvestors` | 外国 VC |\n| `SovereignWealthFundsForeign` | 外国 SWF |\n| `InstitutionsForeignPortfolioInvestorCatergoryOne` | FPI Category 1 |\n| `InstitutionsForeignPortfolioInvestorCatergoryTwo` | FPI Category 2 |\n| `OverseasDepositories` | Overseas Depositories |\n| `OtherInstitutionsForeign` | その他外国機関 |\n| `InstitutionsForeign` | **（再集計）** 外国機関合計 |\n\n### 4.3 個人投資家・その他\n\n| sub_category | 日本語 |\n|---|---|\n| `ResidentIndividualShareholdersHoldingNominalShareCapitalUpToRsTwoLakh` | 国内個人 保有額 ≤ 2 Lakh |\n| `ResidentIndividualShareholdersHoldingNominalShareCapitalInExcessOfRsTwoLakh` | 国内個人 保有額 > 2 Lakh (HNI) |\n| `NonResidentIndians` | NRI |\n| `ForeignNationals` | 外国人個人 |\n| `ForeignCompanies` | 外国企業 |\n| `BodiesCorporate` | 法人 |\n| `OtherNonInstitutions` | その他非機関 |\n| `NonInstitutions` | **（再集計）** 非機関合計 |\n| `InvestorEducationAndProtectionFund` | IEPF |\n\n---\n\n## 5. NonPromoterNonPublic サブカテゴリ\n\n| sub_category | 日本語 |\n|---|---|\n| `CustodianOrDRHolder` | ADR/GDR Custodian |\n| `EmployeeBenefitsTrusts` | 従業員福利厚生信託 |\n\n---\n\n## 6. `is_category_total` フラグの意味\n\n| 値 | 意味 | 特徴 |\n|---|---|---|\n| `1` | 集計行 | shareholder_name が空、num_shareholders は集計数 |\n| `0` | 個別株主明細行 | shareholder_name に個人名 or 法人名 |\n\n集計行 (`is_category_total=1`) の中にも 2 階層:\n- `sub_category=\"\"`: **カテゴリ合計** (例: Promoter 合計 50.11%)\n- `sub_category=\"IndividualsOrHinduUndividedFamily\"`: **sub-category 小計** (例: 0.84%)\n\n---\n\n## 7. Filers の慣習・注意点\n\n### 7.1 創業家は Individuals/HUF にのみ報告される\n\nSEBI 規定では同じ個人を 2 つの sub_category に重複報告しない慣習。\n取締役兼 promoter の創業家は `IndividualsOrHinduUndividedFamily` にのみ現れ、\n`DirectorsAndDirectorsRelatives` が空欄のケースが多い。\n\n**例**:\n- Britannia (Nusli Wadia): hufi に Wadia 家、dir = 0\n- Apollo Tyres (Kanwar): hufi に 3 名、dir = 0\n- Asahi India Glass (Labroo): hufi に 33 名、dir = 0\n\nこの慣習のため、「経営陣に promoter 在籍」判定には `IndividualsOrHinduUndividedFamily >= 1` も\n経営陣兼任の proxy として採用する必要がある。\n\n### 7.2 `OtherIndianShareholders` / `OtherForeignShareholders` の両義性\n\nこれらには複数の本質的に異なる entity が混在し、sub_category だけでは区別不能:\n\n| 混在パターン | 実例 |\n|---|---|\n| 家族 holding company (真の Owner) | PCBL / Rainbow Investments (Goenka)、OLECTRA / MEIL Holdings (Reddy) |\n| 外国多国籍企業 (MNC) | GLAND / Fosun Pharma (中国)、ROUTE / Proximus (ベルギー) |\n| 信託系 (Professional-managed) | TCS / Tata Sons (Tata Trusts 経由) |\n| 政府系 body corporate (filers 誤分類) | IGL / MGL (GAIL, BPCL 系) |\n\n**対処**: `shareholder_name` 詳細行（`is_category_total=0`）の実名を確認する必要あり。\n\n### 7.3 2025-10-31 taxonomy の pct 小数表記\n\n2025-10-31 revision では pct フィールドが小数表記 (0.649 = 64.9%)。\n`market.nse.xbrl._DECIMAL_PCT_TAXONOMIES` で自動 ×100 スケーリング実施済み。\n\n---\n\n## 8. Owner 判定ロジック対応表\n\n各 sub_category が `owner_flag` 判定のどの条件に使用されるか:\n\n| sub_category | owner_flag 判定上の役割 | 関連 Tier |\n|---|---|---|\n| `IndividualsOrHinduUndividedFamily` | `hufi_num` (core シグナル) | Tier 1/3 |\n| `NonResidentIndividualsOrForeignIndividuals` | `nri_num` (NRI family シグナル) | Tier 2 |\n| `DirectorsAndDirectorsRelatives` | `dir_num` (formal 取締役報告) | Tier 1 |\n| `KeyManagerialPersonnel` | `kmp_num` (KMP 報告) | Tier 1 |\n| `RelativesOfPromotersOtherThanPromoterGroup` | `rel_num` (広義親族) | Tier 2 |\n| `TrustsWhereAnyPerson...Trustee...` | `trust_num` (family trust) | Tier 2 |\n| `OtherIndianShareholders` | `other_indian_pct` (holding 経由型) | Tier 3 |\n| `OtherForeignShareholders` | `other_foreign_pct` (foreign holding) | Tier 3 |\n| `ForeignInstitutions`, `ForeignPortfolioInvestor` | `foreign_non_govt_pct` (MNC-JV 判定) | Tier 3 |\n| `CentralGovernmentOrPresidentOfIndia`, `StateGovernmentsOrGovernors`, etc. | `govt_pct` (PSU 除外) | Tier 4 |\n\n### owner_flag ラベル一覧 (notebook 内 `assign_owner_flag()` と一致)\n\n**Tier 1: 高信頼 Owner (AI 不要)**\n\n| ラベル | 条件 |\n|---|---|\n| `owner_confirmed_individual_and_director` | `promoter≥10 AND hufi≥1 AND (dir≥1 OR kmp≥1) AND foreign_non_govt<50` |\n| `owner_confirmed_individual` | `promoter≥10 AND hufi≥1 AND foreign_non_govt<50` |\n| `owner_confirmed_director_only` | `promoter≥10 AND (dir≥1 OR kmp≥1) AND hufi=0 AND nri=0 AND foreign_non_govt<50` |\n\n**Tier 2: 確率中 (AI review 対象)**\n\n| ラベル | 条件 |\n|---|---|\n| `owner_probable_nri_family` | `promoter≥10 AND nri≥1 AND hufi=0` |\n| `owner_probable_relatives_trust` | `promoter≥10 AND (rel≥1 OR trust≥1) AND hufi=0 AND nri=0 AND dir=0 AND kmp=0` |\n\n**Tier 3: 要 AI 判定**\n\n| ラベル | 条件 |\n|---|---|\n| `ambiguous_mnc_jv_candidate` | `promoter≥10 AND hufi≥1 AND foreign_non_govt≥50` |\n| `ambiguous_minor_individual` | `promoter≥10 AND hufi≥1 AND hufi_pct<0.5 AND dir=0 AND kmp=0` |\n| `ambiguous_holding_indian` | `promoter≥10 AND natural_num_sum=0 AND other_indian_pct≥10` |\n| `ambiguous_holding_foreign` | `promoter≥10 AND natural_num_sum=0 AND other_foreign_pct≥10` |\n\n**Tier 4: 除外 (AI 不要)**\n\n| ラベル | 条件 |\n|---|---|\n| `excluded_low_promoter` | `promoter<10` |\n| `excluded_state_dominant` | `govt_pct≥10` |\n| `excluded_no_natural_no_holding` | 上記いずれにも該当せず |\n\n---\n\n## 9. 出典（1 次情報）\n\n| 規則 / 文書 | URL |\n|---|---|\n| SEBI (ICDR) Regulations 2009 | https://www.sebi.gov.in/acts/icdrreg.html |\n| SEBI (ICDR) Regulations 2018 (2025-03-08) | https://www.sebi.gov.in/legal/regulations/mar-2025/securities-and-exchange-board-of-india-issue-of-capital-and-disclosure-requirements-regulations-2018-last-amended-on-march-8-2025-_93559.html |\n| SEBI (SAST) Regulations 2011 Gazette | https://www.sebi.gov.in/sebi_data/attachdocs/1367922725672.pdf |\n| SEBI (LODR) Regulations 2015 Gazette | https://www.sebi.gov.in/sebi_data/attachdocs/1441284401427.pdf |\n| Companies Act 2013 | https://www.indiacode.nic.in/bitstream/123456789/2114/5/A2013-18.pdf |\n| BSE XBRL SHP Taxonomy ZIP | https://www.bseindia.com/downloads1/SHPTaxonomy.zip |\n| NSE Shareholding Pattern 開示様式 | https://nsearchives.nseindia.com/web/sites/default/files/inline-files/Shareholding_Pattern_UR_31%2030062021.pdf |\n\n**本プロジェクトの 1 次情報調査**:\n`research/2026-04-16_nse_promoter_classification/research.md`\n".format(today=_today)

OUT_XBRL_REFERENCE.write_text(_content, encoding="utf-8")
print(f"出力: {OUT_XBRL_REFERENCE}  ({len(_content):,} 文字)")
print(f"  セクション数: 9")


## Section 3: Stage 1 - owner_candidates.csv 生成

緩いフィルタ (`promoter_pct >= 10`) を適用し、Promoter 集計行の各 sub_category から
feature を作成して機械的に `owner_flag` ラベルを付与する。

**出力列**:

- 共通先頭: `symbol, company_name, isin`
- feature: promoter_total_pct, 各 sub_category の num/pct, 集計 (natural_num_sum 等)
- `promoter_names_full_list`: 個別株主名 (is_category_total=0) を `|` 区切り結合
- `owner_flag`: Tier 1〜4 のラベル (機械判定)
- AI 列 (初期空): `owner_flag_ai`, `ai_confidence`, `ai_reasoning`
- `owner_flag_final`: 初期値 OWNER / NOT_OWNER / PENDING_AI_REVIEW


In [ ]:
# Cell 11: Promoter 集計行を pivot
detail_agg = pd.read_sql_query("""
    WITH latest AS (
        SELECT symbol, MAX(report_date) AS rd
        FROM shareholding_detail
        WHERE category='PromoterAndPromoterGroup'
        GROUP BY symbol
    )
    SELECT d.symbol, d.report_date, d.sub_category,
           d.num_shareholders, d.pct_total_shares
    FROM shareholding_detail d
    JOIN latest l ON l.symbol=d.symbol AND l.rd=d.report_date
    WHERE d.category='PromoterAndPromoterGroup' AND d.is_category_total=1
""", conn)

pct = detail_agg.pivot_table(
    index="symbol", columns="sub_category",
    values="pct_total_shares", aggfunc="first", fill_value=0,
)
numsh = detail_agg.pivot_table(
    index="symbol", columns="sub_category",
    values="num_shareholders", aggfunc="first", fill_value=0,
).astype(int)

report_date = detail_agg.groupby("symbol")["report_date"].first()

print(f"Phase 4 取得済み銘柄: {len(pct)}")


In [ ]:
# Cell 12: 個別株主名を集約 (is_category_total=0)
detail_names = pd.read_sql_query("""
    WITH latest AS (
        SELECT symbol, MAX(report_date) AS rd
        FROM shareholding_detail
        WHERE category='PromoterAndPromoterGroup'
        GROUP BY symbol
    )
    SELECT d.symbol, d.shareholder_name
    FROM shareholding_detail d
    JOIN latest l ON l.symbol=d.symbol AND l.rd=d.report_date
    WHERE d.category='PromoterAndPromoterGroup'
      AND d.is_category_total=0
      AND d.shareholder_name != ''
""", conn)

names_per_symbol = (
    detail_names.groupby("symbol")["shareholder_name"]
    .apply(lambda s: NAMES_SEPARATOR.join(s.astype(str).tolist()))
    .rename("promoter_names_full_list")
)
print(f"shareholder_name 詳細行あり: {len(names_per_symbol)} 銘柄")


In [ ]:
# Cell 13: feature 列を構築

NATURAL_SUBS = [
    "IndividualsOrHinduUndividedFamily",
    "NonResidentIndividualsOrForeignIndividuals",
    "DirectorsAndDirectorsRelatives",
    "KeyManagerialPersonnel",
    "RelativesOfPromotersOtherThanPromoterGroup",
    "TrustsWhereAnyPersonBelongingToPromoterAndPromoterGroupIsisTrusteeOrBeneficiaryOrAuthorOfTrust",
]
GOVT_SUBS = [
    "CentralGovernmentOrPresidentOfIndia",
    "StateGovernmentsOrGovernors",
    "ShareholdingByCompaniesOrBodiesCorporatewhereCentralOrStateGovernmentIsPromoter",
    "ForeignGovernment",
    "CentralGovernmentOrStateGovernmentS",
]
FOREIGN_NON_GOVT_SUBS = [
    "ForeignInstitutions",
    "ForeignPortfolioInvestor",
    "OtherForeignShareholders",
]

def _col(df, c):
    return df[c] if c in df.columns else pd.Series(0, index=df.index)

feat = pd.DataFrame(index=pct.index)
feat["report_date"] = report_date
feat["promoter_total_pct"] = _col(pct, "")

# 各 sub_category の num/pct
feat["hufi_num"]  = _col(numsh, "IndividualsOrHinduUndividedFamily")
feat["hufi_pct"]  = _col(pct,   "IndividualsOrHinduUndividedFamily")
feat["nri_num"]   = _col(numsh, "NonResidentIndividualsOrForeignIndividuals")
feat["nri_pct"]   = _col(pct,   "NonResidentIndividualsOrForeignIndividuals")
feat["dir_num"]   = _col(numsh, "DirectorsAndDirectorsRelatives")
feat["dir_pct"]   = _col(pct,   "DirectorsAndDirectorsRelatives")
feat["kmp_num"]   = _col(numsh, "KeyManagerialPersonnel")
feat["kmp_pct"]   = _col(pct,   "KeyManagerialPersonnel")
feat["rel_num"]   = _col(numsh, "RelativesOfPromotersOtherThanPromoterGroup")
feat["rel_pct"]   = _col(pct,   "RelativesOfPromotersOtherThanPromoterGroup")
feat["trust_num"] = _col(numsh, "TrustsWhereAnyPersonBelongingToPromoterAndPromoterGroupIsisTrusteeOrBeneficiaryOrAuthorOfTrust")
feat["trust_pct"] = _col(pct,   "TrustsWhereAnyPersonBelongingToPromoterAndPromoterGroupIsisTrusteeOrBeneficiaryOrAuthorOfTrust")

# 集計値
feat["natural_num_sum"]      = sum(_col(numsh, s) for s in NATURAL_SUBS)
feat["natural_pct_sum"]      = sum(_col(pct, s) for s in NATURAL_SUBS)
feat["other_indian_pct"]     = _col(pct, "OtherIndianShareholders")
feat["other_foreign_pct"]    = _col(pct, "OtherForeignShareholders")
feat["foreign_non_govt_pct"] = sum(_col(pct, s) for s in FOREIGN_NON_GOVT_SUBS)
feat["govt_pct"]             = sum(_col(pct, s) for s in GOVT_SUBS)

# shareholder_names
feat = feat.join(names_per_symbol, how="left")
feat["promoter_names_full_list"] = feat["promoter_names_full_list"].fillna("")

print(f"Features built: {len(feat)} 銘柄 × {len(feat.columns)} 列")
display(feat.describe().round(2))  # noqa: F821


In [ ]:
# Cell 14: owner_flag を優先順位ベースで付与

def assign_owner_flag(row) -> str:
    """優先順位に従って owner_flag ラベルを返す"""
    p      = row["promoter_total_pct"]
    hufi   = row["hufi_num"]
    nri    = row["nri_num"]
    dir_n  = row["dir_num"]
    kmp    = row["kmp_num"]
    rel    = row["rel_num"]
    trust  = row["trust_num"]
    natural= row["natural_num_sum"]
    hufipct= row["hufi_pct"]
    oin    = row["other_indian_pct"]
    ofr    = row["other_foreign_pct"]
    fg     = row["foreign_non_govt_pct"]
    gv     = row["govt_pct"]

    # Tier 4 先判定 (除外)
    if p < MIN_PROMOTER_PCT:
        return "excluded_low_promoter"
    if gv >= MAX_GOVT_DOMINANT_PCT:
        return "excluded_state_dominant"

    # Tier 1
    if hufi >= 1 and (dir_n >= 1 or kmp >= 1) and fg < MAX_FOREIGN_NON_GOVT_PCT:
        return "owner_confirmed_individual_and_director"
    if hufi >= 1 and fg < MAX_FOREIGN_NON_GOVT_PCT:
        return "owner_confirmed_individual"
    if (dir_n >= 1 or kmp >= 1) and hufi == 0 and nri == 0 and fg < MAX_FOREIGN_NON_GOVT_PCT:
        return "owner_confirmed_director_only"

    # Tier 3: MNC-JV
    if hufi >= 1 and fg >= MAX_FOREIGN_NON_GOVT_PCT:
        return "ambiguous_mnc_jv_candidate"

    # Tier 3: 微小 individual
    if hufi >= 1 and hufipct < MIN_MINOR_HUFI_PCT and dir_n == 0 and kmp == 0:
        return "ambiguous_minor_individual"

    # Tier 2
    if nri >= 1 and hufi == 0:
        return "owner_probable_nri_family"
    if (rel >= 1 or trust >= 1) and hufi == 0 and nri == 0 and dir_n == 0 and kmp == 0:
        return "owner_probable_relatives_trust"

    # Tier 3: holding 経由
    if natural == 0 and oin >= MIN_HOLDING_PCT:
        return "ambiguous_holding_indian"
    if natural == 0 and ofr >= MIN_HOLDING_PCT:
        return "ambiguous_holding_foreign"

    return "excluded_no_natural_no_holding"


feat["owner_flag"] = feat.apply(assign_owner_flag, axis=1)

# === act-2026-04-30-004: ハイブリッドルール (既知一族リスト + OWNER_WEAK 降格) ===
# data/config/nse_promoter_classifier.yaml を読み込み、owner_confirmed_director_only に対し
# promoter_names_full_list との照合で再分類:
#   OWNER 一族 → owner_confirmed_director_only (維持)
#   Professional / State / MNC parent → excluded_director_only_<cat> (NOT_OWNER 確定)
#   未マッチ / 矛盾 → ambiguous_director_only_unknown (AI レビューへ)

import re as _re
import yaml as _yaml

_classifier_path = Path("data/config/nse_promoter_classifier.yaml")
_classifier_cfg = _yaml.safe_load(_classifier_path.read_text(encoding="utf-8"))


def _classify_promoter(promoter_names: str) -> str:
    """promoter_names_full_list を 4 カテゴリに分類。
    OWNER / PROFESSIONAL / STATE / MNC / UNKNOWN を返す。"""
    if not isinstance(promoter_names, str) or not promoter_names:
        return "UNKNOWN"
    pn_lower = promoter_names.lower()
    matched_state, matched_mnc, matched_owner_kws, matched_prof = [], [], [], []
    has_override_state = False
    for kw in _classifier_cfg.get("state_keywords", []):
        if kw["keyword"].lower() in pn_lower:
            matched_state.append(kw["keyword"])
    for kw in _classifier_cfg.get("mnc_keywords", []):
        if kw["keyword"].lower() in pn_lower:
            matched_mnc.append(kw["keyword"])
    for kw in _classifier_cfg.get("professional_keywords", []):
        if kw["keyword"].lower() in pn_lower:
            matched_prof.append(kw["keyword"])
    for kw in _classifier_cfg.get("owner_keywords", []):
        if kw["keyword"].lower() in pn_lower:
            matched_owner_kws.append(kw["keyword"])
            if kw.get("override_state"):
                has_override_state = True

    matched_owner = matched_owner_kws

    if matched_state and matched_prof:
        if any("tata sons" in p.lower() for p in matched_prof) and any(
            "president of india" in s.lower() for s in matched_state
        ):
            return "PROFESSIONAL"

    if matched_prof and matched_owner:
        big_inst = ["tata sons", "hdfc bank", "icici bank", "state bank", "bse limited"]
        if any(any(b in p.lower() for b in big_inst) for p in matched_prof):
            return "PROFESSIONAL"
        return "UNKNOWN"

    if matched_state and matched_owner and has_override_state:
        return "OWNER"
    if matched_state:
        return "STATE"
    if matched_mnc:
        return "MNC"
    if matched_prof:
        return "PROFESSIONAL"
    if matched_owner:
        return "OWNER"
    return "UNKNOWN"


_director_only_mask = feat["owner_flag"] == "owner_confirmed_director_only"
_n_director_only = int(_director_only_mask.sum())

if _n_director_only > 0:
    _cls = feat.loc[_director_only_mask, "promoter_names_full_list"].fillna("").map(_classify_promoter)

    def _hybrid_label(c: str) -> str:
        if c == "OWNER":
            return "owner_confirmed_director_only"
        if c == "PROFESSIONAL":
            return "excluded_director_only_professional"
        if c == "STATE":
            return "excluded_director_only_state"
        if c == "MNC":
            return "excluded_director_only_mnc"
        return "ambiguous_director_only_unknown"

    feat.loc[_director_only_mask, "owner_flag"] = _cls.map(_hybrid_label)

    print(f"[hybrid] owner_confirmed_director_only ({_n_director_only}) を再分類:")
    print(_cls.value_counts().to_string())
    print()


# === act-2026-04-30-A2: Tier 1.5 corporate-vehicle rescue ===
# 法人 promoter のみで自然人 promoter が極小・ゼロのケースでも、
# promoter_names_full_list が「既知 Owner 一族 keyword」にマッチすれば
# owner_via_corporate_vehicle として OWNER 救済する。

_rescue_target_flags = {
    "excluded_no_natural_no_holding",
    "excluded_state_dominant",
    "ambiguous_holding_indian",
    "ambiguous_holding_foreign",
    "ambiguous_minor_individual",
}
_rescue_mask = (
    feat["owner_flag"].isin(_rescue_target_flags)
    & (feat["promoter_total_pct"] >= MIN_PROMOTER_PCT)
)
_rescue_n_candidates = int(_rescue_mask.sum())

if _rescue_n_candidates > 0:
    _rescue_cls = feat.loc[_rescue_mask, "promoter_names_full_list"].fillna("").map(_classify_promoter)
    _to_rescue = _rescue_mask & (_rescue_cls.reindex(feat.index, fill_value="UNKNOWN") == "OWNER")
    _n_rescued = int(_to_rescue.sum())
    if _n_rescued > 0:
        feat.loc[_to_rescue, "owner_flag_pre_rescue"] = feat.loc[_to_rescue, "owner_flag"]
        feat.loc[_to_rescue, "owner_flag"] = "owner_via_corporate_vehicle"
        print(f"[corporate-vehicle rescue] {_n_rescued} 銘柄を OWNER 救済:")
        for sym, prev in feat.loc[_to_rescue, "owner_flag_pre_rescue"].items():
            print(f"  {sym}  (was: {prev})")
    else:
        print(f"[corporate-vehicle rescue] 候補 {_rescue_n_candidates} 銘柄のうち救済対象なし")
    print()


# === act-2026-04-30-A3: ambiguous_holding_* の自然人検出救済 ===
# 法人 promoter 主体で family keyword 未マッチ (= A-2 で救済漏れ) でも、
# OtherIndian/ForeignShareholders の個別株主名から自然人を検出できれば
# owner_via_individual_in_other として OWNER 救済する。
#
# 自然人検出パターン:
#   1. HUF 表記: "(HUF)" を含む
#   2. インド敬称: Mr./Mrs./Ms./Shri/Smt/Shrimati/Late/Dr./Prof.
#   3. ALL CAPS で 2-6 単語、かつ法人 suffix (LIMITED/LTD/AB/AG/PTE 等) を含まない
# ※ "Sri Lanka" 等の地名衝突回避のため "Sri" は敬称として採用しない


_HONORIFIC_RE = _re.compile(
    r"\b(Mr\.|Mrs\.|Ms\.|Shri|Smt|Smt\.|Shrimati|Late|Dr\.|Prof\.)\b",
    _re.IGNORECASE,
)
_HUF_RE = _re.compile(r"\(\s*HUF\s*\)", _re.IGNORECASE)
_ALL_CAPS_NAME_RE = _re.compile(r"^[A-Z]+(\s+[A-Z]+){1,5}$")
_CORP_SUFFIXES: set[str] = {
    "LIMITED", "LTD", "LTD.", "PRIVATE", "PVT", "LLP", "INC", "TRUST", "FUND",
    "COMPANY", "HOLDINGS", "GROUP", "INVESTMENTS", "SERVICES",
    "ENTERPRISES", "CORP", "CORPORATION", "CAPITAL", "PARTNERS", "BANK",
    "FOUNDATION", "SOCIETY", "ESTATE", "REALTY", "VENTURES", "TECHNOLOGIES",
    "INDUSTRIES", "COMMERCIAL", "FINANCE", "MANAGEMENT", "SECURITIES",
    "CONSULTANTS", "SOLUTIONS", "SYSTEMS", "PROPERTIES", "ASSOCIATES",
    "AG", "SA", "SE", "GMBH", "PTE", "LLC", "PLC", "BV", "OY", "AB", "NV", "OYJ",
    "INTERNATIONAL", "GLOBAL", "LP", "SAS", "SARL", "BHD", "SDN", "PJSC", "JSC",
    "ASIA", "EUROPE", "AMERICAS", "BIDCO", "TOPCO", "PARENT", "HOLDING",
}


def _is_natural_person(name: str) -> bool:
    """与えられた個別株主名が自然人 (個人) か判定。"""
    if not isinstance(name, str):
        return False
    n = name.strip()
    if not n:
        return False
    if _HUF_RE.search(n):
        return True
    if _HONORIFIC_RE.search(n):
        return True
    if _ALL_CAPS_NAME_RE.match(n):
        words = set(n.split())
        if not (words & _CORP_SUFFIXES):
            return True
    return False


_a3_target_flags = {"ambiguous_holding_indian", "ambiguous_holding_foreign"}
_a3_mask = (
    feat["owner_flag"].isin(_a3_target_flags)
    & (feat["promoter_total_pct"] >= MIN_PROMOTER_PCT)
)

# Other Indian/Foreign Shareholders の個別行から自然人検出
_other_subs = ("OtherIndianShareholders", "OtherForeignShareholders")
_other_indiv = pd.read_sql_query(
    """
    SELECT d.symbol, d.shareholder_name, d.sub_category
    FROM shareholding_detail d
    WHERE d.category='PromoterAndPromoterGroup'
      AND d.is_category_total=0
      AND d.sub_category IN ('OtherIndianShareholders','OtherForeignShareholders')
      AND d.shareholder_name != ''
    """,
    conn,
)

_natural_per_symbol: dict[str, list[str]] = {}
for sym, grp in _other_indiv.groupby("symbol"):
    naturals = [n for n in grp["shareholder_name"] if _is_natural_person(n)]
    if naturals:
        _natural_per_symbol[sym] = naturals

_a3_to_rescue = pd.Series(
    [s in _natural_per_symbol for s in feat.index], index=feat.index
) & _a3_mask
_n_a3_rescued = int(_a3_to_rescue.sum())

if _n_a3_rescued > 0:
    feat.loc[_a3_to_rescue, "owner_flag_pre_rescue"] = feat.loc[_a3_to_rescue, "owner_flag"]
    feat.loc[_a3_to_rescue, "owner_flag"] = "owner_via_individual_in_other"
    print(f"[a3 individual-in-other rescue] {_n_a3_rescued} 銘柄を OWNER 救済:")
    for sym in feat.loc[_a3_to_rescue].index:
        nat_head = _natural_per_symbol[sym][:2]
        print(f"  {sym}  natural_persons={nat_head}")
    print()


# === act-2026-04-30-A1: owner_confirmed_individual のパッシブ識別 ===
# hufi >= 1 だが hufi_pct が極小 (<MIN_MINOR_HUFI_PCT) かつ dir=kmp=0 のケースは、
# SEBI 報告慣習 (持株会社経由保有 + 一族メンバーを Individuals/HUF に名目登記) と
# 純粋投資家一族 (Director でない投資目的家族) の両方が混在する。
#
# 上司定義「経営陣に promoter group の誰かが名を連ねている」の proxy として
# hufi 在籍は維持するが、micro-pct 個人保有は識別ラベル化して目視確認可能にする:
#   owner_confirmed_individual → owner_confirmed_individual_passive (デフォルト OWNER)


_a1_mask = (
    (feat["owner_flag"] == "owner_confirmed_individual")
    & (feat["hufi_pct"] < MIN_MINOR_HUFI_PCT)
    & (feat["dir_num"] == 0)
    & (feat["kmp_num"] == 0)
)
_n_a1 = int(_a1_mask.sum())
if _n_a1 > 0:
    feat.loc[_a1_mask, "owner_flag"] = "owner_confirmed_individual_passive"
    print(f"[a1 passive identification] {_n_a1} 銘柄を passive 識別:")
    for sym in feat.loc[_a1_mask].index:
        print(f"  {sym}  hufi_pct={feat.loc[sym,'hufi_pct']:.3f}")
    print()


feat["ai_review_needed"] = feat["owner_flag"].isin(AI_REVIEW_LABELS)

# AI 結果列 (初期空)
feat["owner_flag_ai"] = ""
feat["ai_confidence"] = pd.NA
feat["ai_reasoning"]  = ""

# owner_flag_final 初期値
def _init_final(flag: str) -> str:
    if flag.startswith("owner_confirmed") or flag in (
        "owner_via_corporate_vehicle",
        "owner_via_individual_in_other",
    ):
        return "OWNER"
    if flag.startswith("excluded"):
        return "NOT_OWNER"
    return "PENDING_AI_REVIEW"

feat["owner_flag_final"] = feat["owner_flag"].map(_init_final)

print("owner_flag 分布:")
print(feat["owner_flag"].value_counts().to_string())
print(f"\nAI review 必要: {feat['ai_review_needed'].sum()} 銘柄")


In [ ]:
# Cell 15: owner_candidates.csv 出力 (symbol, company_name, isin 先頭)
stocks = pd.read_sql_query("SELECT symbol, company_name, isin FROM stocks", conn)

owner_cand = feat.reset_index().merge(stocks, on="symbol", how="left")

COLUMN_ORDER = [
    "symbol", "company_name", "isin",
    "report_date",
    "promoter_total_pct",
    "hufi_num", "hufi_pct",
    "nri_num", "nri_pct",
    "dir_num", "dir_pct",
    "kmp_num", "kmp_pct",
    "rel_num", "rel_pct",
    "trust_num", "trust_pct",
    "natural_num_sum", "natural_pct_sum",
    "other_indian_pct", "other_foreign_pct",
    "foreign_non_govt_pct", "govt_pct",
    "promoter_names_full_list",
    "owner_flag",
    "ai_review_needed",
    "owner_flag_ai",
    "ai_confidence",
    "ai_reasoning",
    "owner_flag_final",
]
owner_cand = owner_cand[COLUMN_ORDER]

owner_cand.to_csv(OUT_OWNER_CANDIDATES, index=False, encoding="utf-8-sig")
print(f"出力: {OUT_OWNER_CANDIDATES}  ({len(owner_cand):,} 行 × {len(owner_cand.columns)} 列)")

print(f"\nowner_flag_final 初期分布:")
display(owner_cand["owner_flag_final"].value_counts())  # noqa: F821

print(f"\nサンプル (5件):")
display(owner_cand.head(5)[["symbol","company_name","isin","promoter_total_pct","hufi_num","owner_flag","ai_review_needed","owner_flag_final"]])  # noqa: F821


## Section 4: (post-AI) owner_flag_final 統合

**前提**: `owner_candidates.csv` の `owner_flag_ai`, `ai_confidence`, `ai_reasoning` が
Claude Code による AI 判定で埋められている。

AI が各銘柄を以下のいずれかに分類:

| AI ラベル         | 意味                                      |
| ----------------- | ----------------------------------------- |
| `ai_owner`        | 真のファミリー企業と判定                  |
| `ai_owner_weak`   | ファミリー要素あるが支配力弱い            |
| `ai_mnc`          | 外国多国籍企業の子会社                    |
| `ai_state`        | 政府/PSU (filers 誤分類含む)              |
| `ai_professional` | Tata Sons 等の信託系 / 機関投資家 managed |
| `ai_inconclusive` | shareholder_name からは判定不能           |

`owner_flag_final` を以下のロジックで確定:

```
owner_confirmed_*    → OWNER
excluded_*           → NOT_OWNER
probable_*/ambiguous_* + ai_owner        → OWNER
probable_*/ambiguous_* + ai_owner_weak   → OWNER_WEAK
probable_*/ambiguous_* + ai_mnc/state/professional → NOT_OWNER
probable_*/ambiguous_* + ai_inconclusive or 空      → PENDING_AI_REVIEW
```


In [ ]:
# Cell 17: AI 結果を読み込んで owner_flag_final を確定

_final = pd.read_csv(OUT_OWNER_CANDIDATES)

def _resolve_final(row) -> str:
    flag = str(row["owner_flag"])
    ai = row.get("owner_flag_ai", "")
    if pd.isna(ai):
        ai = ""
    ai = str(ai).strip().lower()

    if flag.startswith("owner_confirmed") or flag in (
        "owner_via_corporate_vehicle",
        "owner_via_individual_in_other",
    ):
        return "OWNER"
    if flag.startswith("excluded"):
        return "NOT_OWNER"
    # probable_* / ambiguous_*
    if ai == "ai_owner":
        return "OWNER"
    if ai == "ai_owner_weak":
        return "OWNER_WEAK"
    if ai in {"ai_mnc", "ai_state", "ai_professional"}:
        return "NOT_OWNER"
    # 空 or ai_inconclusive
    return "PENDING_AI_REVIEW"


_final["owner_flag_final"] = _final.apply(_resolve_final, axis=1)
_final.to_csv(OUT_OWNER_CANDIDATES, index=False, encoding="utf-8-sig")
print(f"更新: {OUT_OWNER_CANDIDATES}")

print(f"\nowner_flag_final 確定分布:")
display(_final["owner_flag_final"].value_counts())  # noqa: F821

_pending = (_final["owner_flag_final"] == "PENDING_AI_REVIEW").sum()
if _pending > 0:
    print(f"\n⚠️  {_pending} 銘柄が PENDING_AI_REVIEW。AI 判定が未完了か ai_inconclusive。")


## Section 5: 結果確認・統計

最終出力の確認とクロス集計。


In [ ]:
# Cell 19: 統計・サンプル表示
_cand = pd.read_csv(OUT_OWNER_CANDIDATES)

print("=" * 70)
print("owner_flag × owner_flag_final のクロス集計")
print("=" * 70)
display(pd.crosstab(_cand["owner_flag"], _cand["owner_flag_final"]))  # noqa: F821

print("\n" + "=" * 70)
print("OWNER 銘柄 上位 20 (promoter_total_pct 降順)")
print("=" * 70)
owner_only = _cand[_cand["owner_flag_final"] == "OWNER"].sort_values(
    "promoter_total_pct", ascending=False
)
display(owner_only[[  # noqa: F821
    "symbol","company_name","isin","promoter_total_pct","hufi_num","dir_num","owner_flag",
]].head(20))

print("\n" + "=" * 70)
print("PENDING_AI_REVIEW 銘柄 (要 AI 判定)")
print("=" * 70)
pending = _cand[_cand["owner_flag_final"] == "PENDING_AI_REVIEW"]
print(f"件数: {len(pending)}")
if len(pending) > 0:
    print("\nラベル別内訳:")
    display(pending["owner_flag"].value_counts())  # noqa: F821

# クローズ
conn.close()
print("\nDB connection closed.")
print(f"\n=== 全 出力ファイル一覧 ({EXPORT_DIR}) ===")
for p in sorted(EXPORT_DIR.glob("*")):
    sz = p.stat().st_size
    print(f"  {p.name:<40s} {sz:>10,} bytes")


## Section 6: rev1 GT 照合 (ISIN ベース)

`owners.json` (ISIN ベース canonical) と `owner_candidates.csv` を ISIN で JOIN し、
Recall/Precision/F1 を計測。Section 5 までの結果に対する独立検証として実行する。

出力:
- `owners_reconciliation.csv` — ISIN 主キーで両者を結合した照合表
- `rev1_diff_report.md` — FP/FN 集計レポート (act-2026-04-30-007 簡易版)


In [ ]:
# Cell 21: ISIN ベース rev1 GT 照合
# act-2026-04-30-002 (照合ロジックを ISIN ベース化)
import json
from pathlib import Path

OWNERS_JSON = Path("data/cache/nse/owners.json")
_owners_raw = json.loads(OWNERS_JSON.read_text(encoding="utf-8"))

# rev1 canonical: 各 entry は {"company name":..., "isin":..., "Category (...)":...}
_owners_df = pd.DataFrame([
    {
        "isin": (r.get("isin") or "").strip(),
        "company_name_owners": (r.get("company name") or "").strip(),
        "owners_cat": (r.get("Category (Owner, MNC, State, Professional)") or "").strip(),
    }
    for r in _owners_raw
])
# Normalize "state" -> "State"
_owners_df["owners_cat"] = _owners_df["owners_cat"].replace({"state": "State"})
_owners_df = _owners_df[_owners_df["isin"].str.match(r"^IN[A-Z0-9]{9}\d$", na=False)].copy()

print(f"owners.json (rev1 canonical): {len(_owners_df)} rows")
print(f"  Category distribution: {_owners_df['owners_cat'].value_counts().to_dict()}")

# owner_candidates.csv を読み込み (ISIN 主キーで JOIN)
_cand_df = pd.read_csv(OUT_OWNER_CANDIDATES)
_cand_df["isin"] = _cand_df["isin"].fillna("").str.strip()

# LEFT JOIN: owners.json (left) ← owner_candidates.csv (right) on ISIN
_recon = _owners_df.merge(
    _cand_df[["isin", "symbol", "company_name", "owner_flag", "owner_flag_ai", "owner_flag_final"]].rename(columns={"company_name": "company_name_cand"}),
    on="isin",
    how="left",
)

_recon_out = EXPORT_DIR / "owners_reconciliation.csv"
_recon.to_csv(_recon_out, index=False, encoding="utf-8-sig")
print(f"\n書き出し: {_recon_out} ({len(_recon)} rows)")

# Coverage summary
_covered = _recon["symbol"].notna()
print(f"\n=== Coverage ===")
print(f"  intersection (rev1 ∩ candidates): {_covered.sum()} / {len(_recon)}  ({_covered.sum()/len(_recon):.1%})")
print(f"  rev1 only (candidate 未取得): {(~_covered).sum()}")


In [ ]:
# Cell 22: rev1 GT 照合メトリクス算出 (Recall/Precision/F1)
_eval = _recon[_recon["symbol"].notna()].copy()  # intersection only

_eval["is_owner_truth"] = (_eval["owners_cat"] == "Owner")
_eval["is_owner_pred"] = _eval["owner_flag_final"].isin(["OWNER", "OWNER_WEAK"])

_tp = ((_eval["is_owner_truth"]) & (_eval["is_owner_pred"])).sum()
_fp = ((~_eval["is_owner_truth"]) & (_eval["is_owner_pred"])).sum()
_fn = ((_eval["is_owner_truth"]) & (~_eval["is_owner_pred"])).sum()
_tn = ((~_eval["is_owner_truth"]) & (~_eval["is_owner_pred"])).sum()

_precision = _tp / (_tp + _fp) if (_tp + _fp) else 0.0
_recall = _tp / (_tp + _fn) if (_tp + _fn) else 0.0
_f1 = 2*_precision*_recall / (_precision+_recall) if (_precision+_recall) else 0.0

print(f"=== Confusion Matrix (intersection {len(_eval)}) ===")
print(f"  TP={_tp}  FP={_fp}  FN={_fn}  TN={_tn}")
print(f"\n=== Metrics (rev1 ground truth, ISIN-based) ===")
print(f"  Precision: {_precision:.1%}")
print(f"  Recall   : {_recall:.1%}")
print(f"  F1       : {_f1:.1%}")

# FP / FN リスト
_fp_df = _eval[(~_eval["is_owner_truth"]) & (_eval["is_owner_pred"])].copy()
_fp_df[["isin", "company_name_owners", "owners_cat", "symbol", "owner_flag", "owner_flag_ai"]].to_csv(
    EXPORT_DIR / "owners_rev1_false_positives.csv", index=False, encoding="utf-8-sig"
)
print(f"\nFP {len(_fp_df)} 件 -> {EXPORT_DIR}/owners_rev1_false_positives.csv")
if len(_fp_df) > 0:
    print("\nFP 内訳 (owner_flag × owners_cat):")
    display(pd.crosstab(_fp_df["owner_flag"], _fp_df["owners_cat"]))  # noqa: F821

_fn_df = _eval[(_eval["is_owner_truth"]) & (~_eval["is_owner_pred"])].copy()
_fn_df[["isin", "company_name_owners", "symbol", "owner_flag", "owner_flag_ai"]].to_csv(
    EXPORT_DIR / "owners_rev1_false_negatives.csv", index=False, encoding="utf-8-sig"
)
print(f"\nFN {len(_fn_df)} 件 -> {EXPORT_DIR}/owners_rev1_false_negatives.csv")


In [ ]:
# Cell 23: rev1 diff レポート (Markdown) 生成 + 前回比較 (退行検知)
# act-2026-04-30-007: 自動 diff レポート出力
#
# Cell 21-22 の結果を受けて、メトリクス・FP/FN リスト・前回実行との差分を
# 1 つの Markdown ファイル (rev1_diff_report.md) に集約する。
# 既存レポートは rev1_diff_report_prev.md に退避され、新旧比較で
# 新規 FP/FN (退行) と解消 FP/FN を自動検出する。

import re as _re
from datetime import datetime as _dt

REPORT_PATH = EXPORT_DIR / "rev1_diff_report.md"
PREV_REPORT_PATH = EXPORT_DIR / "rev1_diff_report_prev.md"

# 既存 report があれば prev に退避 (前回比較用)
if REPORT_PATH.exists():
    if PREV_REPORT_PATH.exists():
        PREV_REPORT_PATH.unlink()
    REPORT_PATH.rename(PREV_REPORT_PATH)


def _parse_prev_metrics(text: str) -> dict | None:
    """前回 rev1_diff_report.md からメトリクスと FP/FN symbol セットを抽出"""
    if not text:
        return None
    out: dict = {}
    # TP=410 FP=3 FN=0 TN=151 のような形式を抽出
    m = _re.search(r"TP=(\d+).*?FP=(\d+).*?FN=(\d+).*?TN=(\d+)", text)
    if m:
        out["tp"], out["fp"], out["fn"], out["tn"] = (int(g) for g in m.groups())
    # Precision/Recall/F1 (例: "Precision | 99.3%")
    for k in ("precision", "recall", "f1"):
        m = _re.search(rf"{k.capitalize()}\s*\|\s*([\d.]+)%", text)
        if m:
            out[k] = float(m.group(1)) / 100.0
    # FP / FN symbol セット (- [SYMBOL] の形式から抽出)
    fp_section = _re.search(
        r"## 残 FP 一覧\s*\n(.+?)(?=\n## |\Z)", text, _re.DOTALL
    )
    fn_section = _re.search(
        r"## 残 FN 一覧\s*\n(.+?)(?=\n## |\Z)", text, _re.DOTALL
    )
    out["fp_symbols"] = set(_re.findall(r"-\s*\[([A-Z0-9&\-]+)\]", fp_section.group(1))) if fp_section else set()
    out["fn_symbols"] = set(_re.findall(r"-\s*\[([A-Z0-9&\-]+)\]", fn_section.group(1))) if fn_section else set()
    return out


_prev_metrics = None
if PREV_REPORT_PATH.exists():
    _prev_metrics = _parse_prev_metrics(PREV_REPORT_PATH.read_text(encoding="utf-8"))

_current_fp_symbols = set(_fp_df["symbol"].tolist())
_current_fn_symbols = set(_fn_df["symbol"].tolist())


def _diff_section(prev_set: set, curr_set: set, label_new: str, label_fixed: str) -> list[str]:
    out: list[str] = []
    new = curr_set - prev_set
    fixed = prev_set - curr_set
    if new:
        out.append(f"### {label_new} ({len(new)}件)\n")
        for sym in sorted(new):
            out.append(f"- {sym}\n")
        out.append("\n")
    if fixed:
        out.append(f"### {label_fixed} ({len(fixed)}件)\n")
        for sym in sorted(fixed):
            out.append(f"- {sym}\n")
        out.append("\n")
    return out


_lines: list[str] = []
_lines.append("# NSE Owner Candidates - rev1 GT Diff Report\n\n")
_lines.append(f"**生成日時**: {_dt.now().isoformat(timespec='seconds')}\n")
_lines.append(f"**対象**:\n")
_lines.append(f"- owner_candidates.csv: {len(_cand_df)} 銘柄\n")
_lines.append(f"- owners.json (rev1 GT): {len(_owners_df)} 銘柄\n")
_lines.append(f"- intersection: {len(_eval)} 銘柄\n\n")

_lines.append("## メトリクスサマリー\n\n")
_lines.append("| 指標 | 値 |\n")
_lines.append("|---|---|\n")
_lines.append(f"| TP | {_tp} |\n")
_lines.append(f"| FP | {_fp} |\n")
_lines.append(f"| FN | {_fn} |\n")
_lines.append(f"| TN | {_tn} |\n")
_lines.append(f"| Precision | {_precision:.1%} |\n")
_lines.append(f"| Recall | {_recall:.1%} |\n")
_lines.append(f"| F1 | {_f1:.1%} |\n\n")

_lines.append(f"**Confusion**: TP={_tp} FP={_fp} FN={_fn} TN={_tn}\n\n")

# 前回比較
if _prev_metrics:
    _lines.append("## 前回比較\n\n")
    _lines.append("| 指標 | 前回 | 今回 | Δ |\n")
    _lines.append("|---|---|---|---|\n")
    for key, label, fmt in [
        ("tp", "TP", "{:d}"), ("fp", "FP", "{:d}"),
        ("fn", "FN", "{:d}"), ("tn", "TN", "{:d}"),
    ]:
        if key in _prev_metrics:
            curr = locals()[f"_{key}"]
            prev = _prev_metrics[key]
            _lines.append(f"| {label} | {fmt.format(prev)} | {fmt.format(curr)} | {curr-prev:+d} |\n")
    for key, label in [("precision", "Precision"), ("recall", "Recall"), ("f1", "F1")]:
        if key in _prev_metrics:
            curr_v = locals()[f"_{key}"]
            prev_v = _prev_metrics[key]
            _lines.append(f"| {label} | {prev_v:.1%} | {curr_v:.1%} | {(curr_v-prev_v)*100:+.1f}pt |\n")
    _lines.append("\n")

    _lines.extend(_diff_section(
        _prev_metrics["fp_symbols"], _current_fp_symbols,
        "🔴 新規 FP (退行)", "🟢 解消した FP",
    ))
    _lines.extend(_diff_section(
        _prev_metrics["fn_symbols"], _current_fn_symbols,
        "🔴 新規 FN (救済漏れ)", "🟢 解消した FN",
    ))
else:
    _lines.append("## 前回比較\n\n前回レポートなし（初回実行）\n\n")

_lines.append("## 残 FP 一覧\n\n")
if len(_fp_df) > 0:
    for _, r in _fp_df.iterrows():
        _lines.append(
            f"- [{r['symbol']}] {r['company_name_owners']} "
            f"- cat={r['owners_cat']} flag={r['owner_flag']} ai={r.get('owner_flag_ai','') or '-'}\n"
        )
else:
    _lines.append("（なし）\n")
_lines.append("\n")

_lines.append("## 残 FN 一覧\n\n")
if len(_fn_df) > 0:
    for _, r in _fn_df.iterrows():
        _lines.append(
            f"- [{r['symbol']}] {r['company_name_owners']} "
            f"- flag={r['owner_flag']} ai={r.get('owner_flag_ai','') or '-'}\n"
        )
else:
    _lines.append("（なし）\n")
_lines.append("\n")

_lines.append("## 関連ファイル\n\n")
_lines.append(f"- `owners_reconciliation.csv` — ISIN 主キー照合表 ({len(_recon)} rows)\n")
_lines.append(f"- `owners_rev1_false_positives.csv` — FP {len(_fp_df)} 件\n")
_lines.append(f"- `owners_rev1_false_negatives.csv` — FN {len(_fn_df)} 件\n")
_lines.append(f"- `rev1_diff_report_prev.md` — 前回レポート（差分比較用）\n")

REPORT_PATH.write_text("".join(_lines), encoding="utf-8")
print(f"diff レポート生成: {REPORT_PATH}")
print(f"  メトリクス: Precision={_precision:.1%} Recall={_recall:.1%} F1={_f1:.1%}")
if _prev_metrics:
    new_fp = _current_fp_symbols - _prev_metrics['fp_symbols']
    new_fn = _current_fn_symbols - _prev_metrics['fn_symbols']
    if new_fp or new_fn:
        print(f"  ⚠️  退行検知: 新規 FP={sorted(new_fp)}, 新規 FN={sorted(new_fn)}")
    else:
        print(f"  ✅ 退行なし")
